# 08 — Neural Network Regression (MLP)


# ML Course — Practical Notebook
**Target:** `price_etb`  
**Dataset:** `Practical_Dataset for ML.xlsx` (sheet: `Data`)  
**Split:** 80% train / 20% test, `random_state=42`

This notebook is designed for learning and assignment discussion. Run cells from top to bottom. Read the interpretation prompts and write down what your actual results mean. Do not report example or assumed metrics; use the values produced by your run.

## Learning objectives
- Build a feed-forward neural network for a continuous target.
- Understand why scaling both features and target can help neural-network training.
- Use a pipeline and training-only cross-validation for model selection.
- Compare a simple neural network against a baseline.

**Practical note:** This uses scikit-learn's `MLPRegressor`, which is suitable for a tabular regression assignment. It is not a deep-learning framework like PyTorch.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
TARGET = "price_etb"

## 2. Load and inspect the dataset

In [ ]:
possible_paths = [
    Path("data") / "Practical_Dataset for ML.xlsx",
    Path("ML_Course") / "data" / "Practical_Dataset for ML.xlsx",
    Path("..") / "data" / "Practical_Dataset for ML.xlsx",
    Path("Practical_Dataset for ML.xlsx"),
]
DATA_PATH = next((p for p in possible_paths if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Workbook not found. Place 'Practical_Dataset for ML.xlsx' in the project's data/ folder "
        "or set DATA_PATH to its full path."
    )

df = pd.read_excel(DATA_PATH, sheet_name="Data")
print("Loaded:", DATA_PATH)
print("Shape:", df.shape)
display(df.head())

## 3. Prepare features and split the data

We use numeric features only and exclude the target and identifier. Consider removing variables unavailable at the intended prediction time; `days_on_market` can be leakage for an initial listing-price prediction.

In [ ]:
if TARGET not in df.columns:
    raise KeyError(f"Target column {TARGET!r} not found. Available columns: {list(df.columns)}")

excluded = {TARGET, "house_id"}
numeric_features = df.select_dtypes(include=np.number).columns.tolist()
FEATURES = [c for c in numeric_features if c not in excluded]

if not FEATURES:
    raise ValueError("No numeric predictor columns were found.")

X = df[FEATURES].copy()
y = pd.to_numeric(df[TARGET], errors="coerce")
valid = y.notna()
X, y = X.loc[valid], y.loc[valid]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print("Features:", FEATURES)
print(f"Train rows: {len(X_train):,}; test rows: {len(X_test):,}")

## 4. Metrics helper

In [ ]:
def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_true, y_pred),
    }

def evaluate_model(name, estimator, X_train, X_test, y_train, y_test):
    estimator.fit(X_train, y_train)
    train_pred = estimator.predict(X_train)
    test_pred = estimator.predict(X_test)
    row = {
        "Model": name,
        **{f"Train_{k}": v for k, v in regression_metrics(y_train, train_pred).items()},
        **{f"Test_{k}": v for k, v in regression_metrics(y_test, test_pred).items()},
    }
    return row, train_pred, test_pred

## 5. Establish a baseline

A standardized Ridge model is used as a robust linear baseline. It gives context for whether the neural network improves performance.

In [ ]:
baseline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0)),
])
baseline_row, baseline_train_pred, baseline_test_pred = evaluate_model(
    "Ridge baseline", baseline, X_train, X_test, y_train, y_test
)
display(pd.DataFrame([baseline_row]).round(4))

## 6. Build a neural-network pipeline

`TransformedTargetRegressor` standardizes the target during fitting and converts predictions back to ETB. The feature scaler and imputer are fitted only on training data. Early stopping is disabled for compatibility with cross-validation; CV is used to select a compact configuration.

In [ ]:
mlp_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", TransformedTargetRegressor(
        regressor=MLPRegressor(
            max_iter=800,
            early_stopping=False,
            random_state=RANDOM_STATE,
        ),
        transformer=StandardScaler(),
    )),
])

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
param_grid = {
    "model__regressor__hidden_layer_sizes": [(32,), (64,), (64, 32)],
    "model__regressor__alpha": [0.0001, 0.01, 0.1],
    "model__regressor__learning_rate_init": [0.001, 0.01],
}
print("Grid combinations:", len(param_grid["model__regressor__hidden_layer_sizes"]) *
      len(param_grid["model__regressor__alpha"]) *
      len(param_grid["model__regressor__learning_rate_init"]))

## 7. Tune with cross-validation

This grid has 18 configurations × 5 folds. It may take several minutes depending on your computer. For a faster first run, use two hidden-layer choices and one or two alpha values.

In [ ]:
mlp_search = GridSearchCV(
    mlp_pipeline,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
    verbose=1,
)
mlp_search.fit(X_train, y_train)

print("Best parameters:", mlp_search.best_params_)
print(f"Best mean CV MAE: {-mlp_search.best_score_:,.2f} ETB")

cv_results = pd.DataFrame(mlp_search.cv_results_)
display(
    cv_results[[
        "param_model__regressor__hidden_layer_sizes",
        "param_model__regressor__alpha",
        "param_model__regressor__learning_rate_init",
        "mean_train_score", "mean_test_score", "std_test_score"
    ]]
    .assign(mean_train_MAE=lambda d: -d["mean_train_score"],
            mean_CV_MAE=lambda d: -d["mean_test_score"])
    [[
        "param_model__regressor__hidden_layer_sizes",
        "param_model__regressor__alpha",
        "param_model__regressor__learning_rate_init",
        "mean_train_MAE", "mean_CV_MAE", "std_test_score"
    ]]
    .sort_values("mean_CV_MAE").round(4)
)

## 8. Final evaluation on train and test

In [ ]:
best_mlp = mlp_search.best_estimator_
mlp_row, mlp_train_pred, mlp_test_pred = evaluate_model(
    "Tuned Neural Network (MLP)", best_mlp, X_train, X_test, y_train, y_test
)
comparison = pd.DataFrame([baseline_row, mlp_row])
display(comparison.round(4))

print("Interpretation prompts:")
print("1. Does the neural network improve Test_MAE and Test_RMSE over Ridge?")
print("2. Is the training score much better than the test score?")
print("3. Did the model converge? Check warnings and the fitted MLP's n_iter_.")

## 9. Visual diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test, mlp_test_pred, alpha=0.65)
lo = min(float(y_test.min()), float(np.min(mlp_test_pred)))
hi = max(float(y_test.max()), float(np.max(mlp_test_pred)))
axes[0].plot([lo, hi], [lo, hi], linestyle="--")
axes[0].set_xlabel("Actual price (ETB)")
axes[0].set_ylabel("Predicted price (ETB)")
axes[0].set_title("Neural network: actual vs predicted")

residuals = y_test.to_numpy() - np.asarray(mlp_test_pred)
axes[1].scatter(mlp_test_pred, residuals, alpha=0.65)
axes[1].axhline(0, linestyle="--")
axes[1].set_xlabel("Predicted price (ETB)")
axes[1].set_ylabel("Residual (actual - predicted)")
axes[1].set_title("Neural network: residuals")

plt.tight_layout()
plt.show()

fitted_mlp = best_mlp.named_steps["model"].regressor_
print("Training iterations used:", fitted_mlp.n_iter_)
print("Final training loss (target-standardized scale):", fitted_mlp.loss_)

## 10. Save outputs

In [ ]:
output_dir = Path("outputs") / "metrics"
output_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame([baseline_row]).to_csv(output_dir / "08_neural_network_baseline.csv", index=False)
comparison.to_csv(output_dir / "08_neural_network_final_metrics.csv", index=False)
cv_results.to_csv(output_dir / "08_neural_network_cv_results.csv", index=False)

print("Saved neural-network results to:", output_dir.resolve())

## 11. What to explain to your teacher

- An MLP is a feed-forward neural network with hidden layers and nonlinear activation functions.
- Standardization helps gradient-based optimization; the target is standardized during training and predictions are returned to ETB.
- The `alpha` parameter controls L2 regularization on network weights.
- Cross-validation chooses the configuration using training data; the held-out test set is reserved for final evaluation.
- Neural networks are not guaranteed to beat simpler models on tabular data. Compare test metrics and consider runtime, stability, and interpretability.
- For a production-quality experiment, use the same rows, features, and split across all model notebooks, and remove any features unavailable at prediction time.

### Understanding check (answer before looking up anything)
1. What does a hidden layer learn?
2. Why do we standardize features for an MLP?
3. What does `alpha` control?
4. Why is the neural network not automatically the best model?